In [32]:
import logging
from pathlib import Path

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "rag_pipeline.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("RAG")

In [33]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from pgvector.psycopg2 import register_vector
import psycopg2
import ollama

# ==========================================================
# Configuration
# ==========================================================

HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"

TABLE = "rag_chunks"

EMBED_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
OLLAMA_MODEL = "qwen3:8b"
print("Loading models...")

embed_model = SentenceTransformer(EMBED_MODEL)

reranker = CrossEncoder(RERANK_MODEL)

print("Models loaded.\n")


def data_retrival(
    SEARCH_QUERY,
    QUESTION,
    RULES,
    TOP_K_RETRIEVAL=10,
    TOP_K_FINAL=5,
    TEMPERATURE=0.1,
    TOP_P=0.9,
    TOP_K=10,
):



    # ==========================================================
    # Create Query Embedding
    # ==========================================================

    query_embedding = embed_model.encode(
        SEARCH_QUERY,
        normalize_embeddings=True
    ).tolist()

    embedding_str = "[" + ",".join(map(str, query_embedding)) + "]"

    # ==========================================================
    # PostgreSQL Connection
    # ==========================================================

    logger.info("\nConnecting PostgreSQL...")

    conn = psycopg2.connect(
        host=HOST,
        port=PORT,
        user=USER,
        password=PASSWORD,
        dbname=DATABASE
    )

    register_vector(conn)

    cur = conn.cursor()

    # ==========================================================
    # Semantic Search
    # ==========================================================

    sql = f"""
    SELECT
        page_content,
        source,
        metadata,
        embedding <=> %s::vector AS distance
    FROM {TABLE}
    ORDER BY distance
    LIMIT %s;
    """

    cur.execute(
        sql,
        (
            embedding_str,
            TOP_K_RETRIEVAL
        )
    )

    rows = cur.fetchall()

    cur.close()
    conn.close()

    if not rows:
        logger.info("No documents found.")
        return "No documents found."
    


    logger.info("\n" + "=" * 80)
    logger.info("TOP VECTOR SEARCH RESULTS")
    logger.info("=" * 80)

    for i, row in enumerate(rows, start=1):

        page_content, source, metadata, distance = row

        logger.info(f"\nRank : {i}")
        logger.info(f"Distance : {distance:.5f}")
        logger.info(f"Source : {source}")
        logger.info("-" * 80)
        logger.info(page_content[:300])

    # ==========================================================
    # Cross Encoder Reranking
    # ==========================================================
    
    rerank_query = f"""
    Search Query:
    {SEARCH_QUERY}

    User Question:
    {QUESTION}
    """

    pairs = [
        (rerank_query, row[0])
        for row in rows
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for row, score in zip(rows, scores):

        page_content, source, metadata, distance = row

        reranked.append({
            "page_content": page_content,
            "source": source,
            "metadata": metadata,
            "distance": distance,
            "rerank_score": float(score)
        })

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    reranked = reranked[:TOP_K_FINAL]
    logger.info("\n" + "=" * 80)
    logger.info("TOP RERANKED CHUNKS")
    logger.info("=" * 80)

    for i, doc in enumerate(reranked, start=1):

        logger.info(f"\nRank : {i}")
        logger.info(f"Distance : {doc['distance']:.5f}")
        logger.info(f"Reranker : {doc['rerank_score']:.5f}")
        logger.info(f"Source : {doc['source']}")
        logger.info("-" * 80)

        logger.info(doc["page_content"][:500])

    logger.info("\n" + "=" * 80)

    # ==========================================================
    # Build Context
    # ==========================================================

    context = "\n\n".join(
        doc["page_content"]
        for doc in reranked
        )

    # ==========================================================
    # Build Rules
    # ==========================================================

    STANDARD_RULES = [
        "Use ONLY the supplied context.",
        "Do NOT use outside knowledge.",
        "Return only the requested information.",
        "Do not explain unless requested.",
        "Do not summarize unless requested.",
        "If the answer appears explicitly in the context, return it exactly."
        "If the context partially answers the question, return the available information."
        "Only reply 'Data is not available.' when no relevant information exists."
    ]

    ALL_RULES = STANDARD_RULES + RULES

    logger.info("\n" + "=" * 80)
    logger.info("RULES")
    logger.info("=" * 80)

    for i, rule in enumerate(ALL_RULES, start=1):
        logger.info(f"{i}. {rule}")

    rule_text = ""

    for i, rule in enumerate(ALL_RULES, start=1):
        rule_text += f"{i}. {rule}\n"

    # ==========================================================
    # Prompt
    # ==========================================================

    prompt = f"""
You are a RAG assistant.

Answer ONLY from the supplied context.

Context:
{context}

Question:
{QUESTION}

Rules:
{rule_text}

Answer:
"""
    # ==========================================================
    # Ask Ollama
    # ==========================================================

    logger.info("\nGenerating Answer...\n")

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # ==========================================================
    # Final Output
    # ==========================================================

    answer = response["message"]["content"]

    # Remove Qwen thinking block if present
    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1].strip()

    logger.info("=" * 80)
    logger.info("FINAL ANSWER")
    logger.info("=" * 80)
    logger.info(answer)

    logger.info("\n" + "=" * 80)
    logger.info("RETRIEVAL STATISTICS")
    logger.info("=" * 80)
    logger.info(f"Vector Search Top-K : {TOP_K_RETRIEVAL}")
    logger.info(f"Reranker Top-K      : {TOP_K_FINAL}")
    logger.info(f"Context Chunks      : {len(reranked)}")
    logger.info(f"Prompt Characters   : {len(prompt)}")
    logger.info("=" * 80)

    with open("../console.md", "w", encoding="utf-8") as f:
        f.write(answer)

    return answer

2026-07-16 01:20:13,901 | INFO     | No device provided, using cpu


Loading models...


2026-07-16 01:20:14,232 | INFO     | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 01:20:14,242 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-07-16 01:20:14,537 | INFO     | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 01:20:14,551 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-16 01:20:14,552 | INFO     | Loading SentenceTransformer model from BAAI/bge-m3.
2026-07-16 01:20:14,765 | INFO     | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 01:20:14

Models loaded.



In [34]:
import json
import ollama


def prepare_question(question):
    """
    Generate:
        1. Optimized semantic search query
        2. Question-specific answer rules
    """

    planner_prompt = f"""
You are an expert Query Planner for a Retrieval-Augmented Generation (RAG) system.

Your job is NOT to answer the user's question.

Your ONLY job is to prepare retrieval instructions.

=====================================================================

USER QUESTION

{question}

=====================================================================

TASK 1

Generate the BEST semantic search query.

Guidelines:

- Preserve technical terminology exactly.
- Preserve product names.
- Preserve menu names.
- Preserve UI labels.
- Preserve figure names.
- Preserve table names.
- Preserve chapter names.
- Preserve command names.
- Preserve important noun phrases.
- Remove conversational words.
- Remove answer formatting instructions.
- Keep the search query concise.
- Maximum 12 words.

=====================================================================

TASK 2

Generate ONLY question-specific answer rules.

Examples of GOOD rules:


- Preserve the original numbering.
- Return the complete procedure.
- Preserve the original wording.

DO NOT generate:

- Generic RAG rules.
- Explanations.
- New questions.
- Configuration steps.
- Hallucinated information.

=====================================================================

Return ONLY valid JSON.

Expected JSON format:

{{
    "search_query": "...",
    "rules": [
        "...",
        "...",
        "..."
    ]
}}

Example

User Question:

Give me the table of Hot Keys in the Operator Workplace.
Print the output in table format.
Skip first 7 rows.

Expected Output:

{{
    "search_query": "Hot Keys, Operator Workplace",

    "rules": [
        "Return the answer in table format.",
        "Skip the first 7 rows.",
        "Preserve the original row order."
    ]
}}

IMPORTANT

Return ONLY JSON.

Do NOT return Markdown.

Do NOT wrap the JSON inside ```json.

Do NOT explain your reasoning.

Do NOT include <think>.
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": planner_prompt
            }
        ]
    )

    text = response["message"]["content"].strip()

    # ----------------------------------------------------------
    # Remove <think>...</think> (Qwen3 sometimes generates this)
    # ----------------------------------------------------------

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    # ----------------------------------------------------------
    # Remove Markdown code fences
    # ----------------------------------------------------------

    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    # ----------------------------------------------------------
    # Parse JSON
    # ----------------------------------------------------------

    try:
        result = json.loads(text)

    except json.JSONDecodeError:

        logger.info("=" * 80)
        logger.info("INVALID JSON RETURNED BY PLANNER")
        logger.info("=" * 80)
        logger.info(text)

        raise

    # ----------------------------------------------------------
    # Print Planner Output
    # ----------------------------------------------------------

    logger.info("\n" + "=" * 80)
    logger.info("SEARCH QUERY")
    logger.info("=" * 80)
    logger.info(result["search_query"])

    logger.info("\n" + "=" * 80)
    logger.info("QUESTION SPECIFIC RULES")
    logger.info("=" * 80)

    for rule in result["rules"]:
        logger.info(f"- {rule}")

    logger.info("-------")

    return result["search_query"], result["rules"]

In [36]:


QUESTION = """
How to structure the diagram?
"""

SEARCH_QUERY, RULES = prepare_question(QUESTION)

logger.info("=" * 80)
logger.info("SEARCH QUERY")
logger.info("=" * 80)
logger.info(SEARCH_QUERY)

logger.info("**************")

logger.info("=" * 80)
logger.info("QUESTION SPECIFIC RULES")
logger.info("=" * 80)

for rule in RULES:
    print("-", rule)


answer = data_retrival(
    SEARCH_QUERY=SEARCH_QUERY,
    QUESTION=QUESTION,
    RULES=RULES
)

with open("../console.md", "w", encoding="utf-8") as f:
        f.write(answer)

logger.info(f" Answer:- {answer}")

2026-07-16 01:24:45,052 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-07-16 01:24:45,054 | INFO     | 
2026-07-16 01:24:45,054 | INFO     | SEARCH QUERY
2026-07-16 01:24:45,055 | INFO     | ================================================================================
2026-07-16 01:24:45,055 | INFO     | Diagram Structure
2026-07-16 01:24:45,056 | INFO     | 
2026-07-16 01:24:45,056 | INFO     | QUESTION SPECIFIC RULES
2026-07-16 01:24:45,056 | INFO     | ================================================================================
2026-07-16 01:24:45,057 | INFO     | - Preserve the original numbering.
2026-07-16 01:24:45,057 | INFO     | - Return the complete procedure.
2026-07-16 01:24:45,057 | INFO     | - Preserve the original wording.
2026-07-16 01:24:45,058 | INFO     | -------
2026-07-16 01:24:45,058 | INFO     | ================================================================================
2026-07-16 01:24:45,059 | INFO     | SEAR

- Preserve the original numbering.
- Return the complete procedure.
- Preserve the original wording.


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.89it/s]
2026-07-16 01:24:45,140 | INFO     | 
Connecting PostgreSQL...
2026-07-16 01:24:45,168 | INFO     | 
2026-07-16 01:24:45,169 | INFO     | TOP VECTOR SEARCH RESULTS
2026-07-16 01:24:45,170 | INFO     | ================================================================================
2026-07-16 01:24:45,170 | INFO     | 
Rank : 1
2026-07-16 01:24:45,170 | INFO     | Distance : 0.38157
2026-07-16 01:24:45,171 | INFO     | Source : ABB 800xA.pdf
2026-07-16 01:24:45,172 | INFO     | --------------------------------------------------------------------------------
2026-07-16 01:24:45,172 | INFO     | The structure diagram in the 800xA System is used for visual tracking of sequential control processes and to display details of steps, actions, and transitions. The general display of the sequence control is based on the IEC 1131-3 standards. The default colors for the SFC Viewer animation can be ch
2026-07-16 01:24:45,173 | INFO     | 
Rank :